---
title: "Review Gates and Interrupts"
draft: true
categories: [agents, workflows, langgraph]
---


Human review is a workflow transition. The interrupt payload must contain the artifact, unresolved issues, and allowed decisions; the response must be validated before it can affect state.

## Pause and resume one thread

The review node contains no side effect. `interrupt()` pauses it, and `Command(resume=...)` supplies the typed decision when the same thread resumes.


In [1]:
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
from evidence_brief.fixtures import request_for
from evidence_brief.workflow import build_evidence_brief_graph, make_context

def start_thread(thread_id):
    graph = build_evidence_brief_graph(checkpointer=InMemorySaver())
    context = make_context()
    config = {"configurable": {"thread_id": thread_id}}
    paused = graph.invoke(
        {"request": request_for("conflict-01").model_dump(), "events": [], "branch_results": []},
        config=config, context=context, version="v2",
    )
    return graph, context, config, paused

graph, context, config, paused = start_thread("approve")
request = paused.interrupts[0].value
print({"recommendation": request["recommendation"], "actions": request["allowed_actions"]})
approved = graph.invoke(
    Command(resume={"action": "approve", "reason": "evidence reviewed"}),
    config=config, context=context, version="v2",
)
print(approved.value["status"], context.controller.effects)
assert approved.value["status"] == "complete"
assert context.controller.effects[-1] == "export:artifact"


{'recommendation': 'pilot_only', 'actions': ['approve', 'edit', 'reject', 'request_evidence']}
complete ['collect:security', 'collect:performance', 'collect:operations', 'export:artifact']


Collection effects occur before review and export occurs after approval. Resuming restarts the review node, but there is nothing in that node to duplicate.

## Decisions change control flow

Malformed input creates a fresh interrupt, rejection performs one bounded revision, evidence requests revisit research, and edits change the exported artifact rather than merely adding a comment.


In [2]:
def resume_sequence(thread_id, decisions):
    graph, context, config, result = start_thread(thread_id)
    for decision in decisions:
        result = graph.invoke(Command(resume=decision), config=config, context=context, version="v2")
    return result, context

malformed, _ = resume_sequence("malformed", [
    {"action": "unknown", "reason": "invalid"},
])
edited, edit_context = resume_sequence("edit", [
    {"action": "edit", "reason": "narrow scope", "edited_recommendation": "pilot_with_residency_gate"},
])
revised, _ = resume_sequence("reject", [
    {"action": "reject", "reason": "make risk explicit"},
    {"action": "approve", "reason": "revision accepted"},
])
more_evidence, _ = resume_sequence("more-evidence", [
    {"action": "request_evidence", "reason": "repeat the bounded collection"},
])

print({
    "malformed_reinterrupts": len(malformed.interrupts),
    "edited_recommendation": edited.value["artifact"]["recommendation"],
    "revision_count": revised.value["revision_count"],
    "evidence_request_reinterrupts": len(more_evidence.interrupts),
})
assert malformed.interrupts and more_evidence.interrupts
assert edited.value["artifact"]["recommendation"] == "pilot_with_residency_gate"
assert revised.value["revision_count"] == 1
assert edit_context.controller.effects.count("export:artifact") == 1


{'malformed_reinterrupts': 1, 'edited_recommendation': 'pilot_with_residency_gate', 'revision_count': 1, 'evidence_request_reinterrupts': 1}


The interrupt protocol now has meaningful semantics: approval advances, edits alter versioned state, rejection revises, and an evidence request revisits research. Durable process restart is a persistence concern and is tested in Chapter 07.
